In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1
    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [1]

labels_str = ['Benign', 'DDOS attack-HOIC', 'DDoS', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[1]

# ROC

In [4]:
from sklearn.metrics import roc_curve
exp_test = []
y_true_all_exp_test = []
for i in range(len(filenames)):
    exp_test.append(pd.read_csv(f'test_{filenames[i]}_GMM_BIC_1_10_scores.csv'))
    y_true_all_exp_test.append(exp_test[i]['Label'].values.tolist())
    exp_test[i] = exp_test[i].drop(columns=['Label'])


In [5]:
from sklearn.metrics import classification_report
ths = [21]
f1s = [[]]
cd_test_index = []
for i in range(len(exp_test)):
    index = 0
    cd_test_index.append([])
    for th in ths:
        y_pred = []
        for j in range(len(exp_test[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_test[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_test[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
                cd_test_index[-1].append(j)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_test[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_test[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, 6, -1]
th = 21 hidden = 1


,0,1,2,3,4,5,6,-1
0,156741,0,575,10,2,233,271,2838
1,0,0,0,0,0,0,0,137202
2,357,0,115165,0,0,0,0,62
3,0,0,0,102358,0,0,0,523
4,0,0,0,1,76127,0,0,59
5,18,0,2,0,0,57077,0,141
6,9,0,2,0,0,8,142,24
-1,0,0,0,0,0,0,0,0


th: 21 hidden: 1 acc:0.9943887732384333 recall:1.0 precision:0.9741070224140747 f1:0.986883701191508
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.97      1.00      0.99    137202
           0       1.00      0.98      0.99    160670
           2       0.99      1.00      1.00    115584
           3       1.00      0.99      1.00    102881
           4       1.00      1.00      1.00     76187
           5       1.00      1.00      1.00     57238
           6       0.34      0.77      0.47       185

    accuracy                           0.99    649947
   macro avg       0.90      0.96      0.92    649947
weighted avg       0.99      0.99      0.99    649947



In [6]:
cd_label = pd.DataFrame(y_true_all_exp_test[0], columns=['Label']).loc[cd_test_index[0]]
cd_label[cd_label['Label'] == 1].index

Index([     0,      4,      6,     10,     22,     25,     40,     43,     44,
           45,
       ...
       649923, 649927, 649929, 649932, 649933, 649934, 649938, 649939, 649943,
       649944],
      dtype='int64', length=132260)

In [7]:
exp_test[0].loc[cd_label[cd_label['Label'] == 1].index].idxmax(axis=1).value_counts()

0    126766
2      5494
Name: count, dtype: int64

# Média absolute threshold

In [9]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 20: 0.49481646297955945


# CD detectados

In [6]:
display(pd.DataFrame(cd_test_index))

,0,1,2,3,4,5,6,7,8,9,...,1496,1497,1498,1499,1500,1501,1502,1503,1504,1505
0,158,550,570,698,787,922,1032,1243,1404,1700,...,702410,702836,703413,704406,705869,706122,707139,707160,708749,710023


In [12]:
train = pd.read_csv('../../../train_CADE_cod_joined_attacks.csv')
test = pd.read_csv('../../../test_CADE_cod_joined_attacks.csv')

In [18]:
for i in range(len(filenames)):
    cd_test = test.loc[cd_test_index[i]]
    n_samples = test[test['Label'] == labels_str[filenames[i]]].shape[0]
    print(cd_test['Label'].value_counts())
    cd_test['Label'] = 'New Concept'
    retrain = pd.concat([train,cd_test], ignore_index=True)
    new_test = test.drop(cd_test_index[i])
    new_samples = train[train['Label'] == labels_str[filenames[i]]].sample(frac=n_samples/train[train['Label'] == labels_str[filenames[i]]].shape[0], random_state=123)
    new_test = pd.concat([new_test,new_samples],ignore_index=True)
    print(retrain['Label'].value_counts())
    print(new_test['Label'].value_counts())
    retrain.to_csv(f'retrain_hidden_{filenames[i]}_CADE_cod_joined_attacks.csv',index=False)
    new_test.to_csv(f'retest_hidden_{filenames[i]}_CADE_cod_joined_attacks.csv',index=False)

Label
DDoS             185839
Benign            14778
Infilteration      5366
DoS                 460
Bot                 193
BruteForce           47
Web                  13
Name: count, dtype: int64
Label
DDoS             808918
Benign           514148
DoS              418752
BruteForce       243804
New Concept      206696
Bot              183163
Infilteration    102810
Web                 595
Name: count, dtype: int64
Label
DDoS             319733
Benign           145892
DoS              130400
BruteForce        76141
Bot               57045
Infilteration     26761
Web                 172
Name: count, dtype: int64
Label
DoS              100917
Benign            25230
Infilteration      7484
DDoS                553
Bot                 114
Web                  14
BruteForce           14
Name: count, dtype: int64
Label
DDoS             808918
Benign           514148
DoS              418752
BruteForce       243804
Bot              183163
New Concept      134326
Infilteration    102810
We